# model-check template

The standard structure for a `model_checks/` notebook. Fill the four config cells — **SPACE**,
**TARGET**, **SEGMENTER**, **MODEL** — and the Summary cell runs the three gates and writes one row to
`results.md`. Rules of the house:

- **one segmentation method per notebook.** A different method is a different notebook, not a variant.
- **two spaces, set in two places.** SPACE = the MODEL space (what the model reads). The SEGMENTER
  cell opens with `SEGMENT_FEATURES` = the SEGMENTATION space, defaulting to `FEATURES`; override it to
  segment on different variables than the model. Both may be 2D or 4D — the code is dimension-agnostic.
- **`keep_segment(keep=...)`** picks which segments to bet: `"profitable"` (all positive-edge) or
  `"best"` (the single highest-edge one); swap via `kmeans_segmenter(..., keep=...)`.
- **the SEGMENTER cell owns its own segment-edge print** (a self-description of the method).
- **extra checks go below the Summary**, outside the template.

Copy this file, rename it `{space}_{target}_{segmenter}_{model}`, and edit the four cells.

In [ ]:
# --- SETUP -------------------------------------------------------------------
GROUP = "major"   # league group: which <group>/ ABT to validate
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline

from evaluation.model_check import run_check, collect_bets, pooled_roi, roi_grouped, season_folds

abt = pd.read_parquet(f"C:/Users/twluc/frame/01_data/03_abt/{GROUP}/abt.parquet")

In [ ]:
# --- SPACE (modifiable) — the model space, defined in one place --------------
# The MODEL space: the features the bet-signal model reads. This is the ONLY cell to change
# to move the model to another space. (The segmentation space is set in the SEGMENTER cell.)
FEATURES = [
    "hmt_season_goals_for_avg",
    "hmt_season_goals_agst_avg",
    "awt_season_goals_for_avg",
    "awt_season_goals_agst_avg",
]

In [ ]:
# --- TARGET (modifiable) -----------------------------------------------------
# OUTCOME is the 0/1 event we bet on. IMPLIED is the market-consensus implied probability
# (mrkt), used for both the profitability check and the ROI — one odds source, never mixed.
OUTCOME = "t_draw_flg"          # 1 when the match ended in a draw
IMPLIED = "mrkt_draw_impl"      # implied draw prob = 1 / mrkt draw odds (profitability + betting)

In [ ]:
# --- ABT FILTERING -----------------------------------------------------------
# Keep matches where both teams have enough history for stable season averages; drop rows missing any
# feature or the target.
MIN_GAME_NUMBER = 8
matches = abt[
    (abt["hmt_game_number"] > MIN_GAME_NUMBER)
    & (abt["awt_game_number"] > MIN_GAME_NUMBER)
].dropna(subset=FEATURES + [IMPLIED, OUTCOME, "league", "season"]).copy()

print(f"matches: {len(matches)}   seasons: {sorted(matches['season'].unique())}   "
      f"leagues: {sorted(matches['league'].unique())}")

In [ ]:
# --- SEGMENTER (modifiable) — KMeans on the running space --------------------
# SEGMENT_FEATURES is the space the SEGMENTATION uses. It defaults to the model's FEATURES (segment on
# the same variables the model sees); override it with a different list to segment on other variables
# than the model. The model space (FEATURES, used by run_check below) is unaffected.
SEGMENT_FEATURES = FEATURES

# The method IS this notebook's identity; a different method = a different notebook. Here: fit KMeans on
# SEGMENT_FEATURES (train only), keep the chosen clusters (keep="profitable" or "best"), label each row
# by its kept cluster (NaN outside -> not bet). Contract: segmenter(train) -> labeler ; labeler -> Series.

# Per-segment edge on the TRAIN fold only, so the held-out fold never influences the choice.
#   keep="profitable" -> keep every positive-edge segment;   keep="best" -> keep only the single
# highest-edge one. Swap it via the `keep=` argument of kmeans_segmenter below.
def keep_segment(train, labels, keep="profitable", min_matches=100):
    frame = pd.DataFrame({"label": labels.to_numpy(),
                          "won": train[OUTCOME].to_numpy(),
                          "implied": train[IMPLIED].to_numpy()})
    stats = frame.groupby("label").agg(won=("won", "mean"), implied=("implied", "mean"), n=("won", "size"))
    edge = (stats["won"] - stats["implied"])[stats["n"] >= min_matches]
    positive = edge[edge > 0]
    if keep == "best":
        return {positive.idxmax()} if len(positive) else set()   # single highest-edge segment
    return set(positive.index)                                    # all positive-edge segments

def kmeans_segmenter(features, n_clusters, keep="profitable", min_matches=100):
    def segmenter(train):
        km = KMeans(n_clusters=n_clusters, random_state=0, n_init=10).fit(train[features].to_numpy())
        labels = pd.Series(km.predict(train[features].to_numpy()), index=train.index)
        kept = keep_segment(train, labels, keep=keep, min_matches=min_matches)
        def labeler(rows):
            c = pd.Series(km.predict(rows[features].to_numpy()), index=rows.index)
            return c.where(c.isin(kept))     # non-kept clusters -> NaN -> not bet
        return labeler
    return segmenter

K = 5   # cluster count (see the k-sweep extra check below)
segmenter = kmeans_segmenter(SEGMENT_FEATURES, K)   # keep="best" to bet only the single best cluster

# The segmenter's self-description: fit on all matches (in-sample) and show each cluster's edge — size,
# outcome rate vs implied, ROI, centre (in SEGMENT_FEATURES). In-sample: a map of WHERE the edge sits,
# not gated evidence. KEEP marks clusters a training fold would keep (positive edge).
_km = KMeans(n_clusters=K, random_state=0, n_init=10).fit(matches[SEGMENT_FEATURES].to_numpy())
_lab = _km.predict(matches[SEGMENT_FEATURES].to_numpy())
print(f"segment edges (in-sample, k={K})   centre = [{'  '.join(SEGMENT_FEATURES)}]")
print(" seg      n   rate%  impl%   edge     ROI   keep   centre")
for s in range(K):
    m = matches[_lab == s]
    dr, im = m[OUTCOME].mean(), m[IMPLIED].mean()
    centre = " ".join(f"{x:.2f}" for x in _km.cluster_centers_[s])
    print(f" {s:>3} {len(m):>7}  {dr*100:5.1f}  {im*100:5.1f}  {(dr-im)*100:+5.1f}  "
          f"{pooled_roi(m[IMPLIED], m[OUTCOME].astype(int)):+6.3f}  {'KEEP' if dr > im else '    '}  [{centre}]")

In [ ]:
# --- MODEL (modifiable) ------------------------------------------------------
# The bet-signal model: any sklearn-style estimator with .fit(X, y) and .predict_proba(X). It is fit
# on the MODEL space (FEATURES) — independent of the segmentation space above.
def make_model():
    return make_pipeline(PolynomialFeatures(degree=2, include_bias=False),
                         StandardScaler(), LogisticRegression(max_iter=3000))

BUFFER = 0.02   # only bet when the model probability beats the implied one by at least this margin

## Summary
The three gates at a glance, and the one-line row written to `model_checks/results.md`.

In [ ]:
_ = run_check(matches, features=FEATURES, outcome=OUTCOME, implied=IMPLIED,
              segmenter=segmenter, make_model=make_model, buffer=BUFFER,
              source="REPLACE_WITH_NOTEBOOK_NAME", space="SPACE", target="draw",
              abt_filter="major, gn>8, dropna",
              segmentation=f"KMeans k={K} (match, running, {len(SEGMENT_FEATURES)}D)",
              segment_filter="train-profitable clusters")

## Extra checks — below the template
Everything above is the standard template (config cells + Summary). Cells below are notebook-specific
diagnostics, outside the template — they may use knobs a given segmenter has (e.g. a cluster count)
that other methods do not.